# Few-Shot Day-to-Night Translation — Colab Smoke Test

This notebook validates one short img2img-turbo training run on Colab. It is **not** the recommended entry point for the full 18-run experiment matrix: managed Colab runtimes can terminate before long jobs finish.

The notebook deliberately:

- creates the project's pinned Python 3.10 environment inside Miniforge;
- checks out the pinned official img2img-turbo revision through the project setup script;
- copies one compressed reduced dataset from Drive to the Colab VM before extracting it;
- runs a dry-run before using the GPU;
- writes checkpoints and logs directly to Google Drive.

Recommended reduced archive contents for the default 10-shot smoke test:

```text
data/processed/day2night/train/day/       # the selected training images
data/processed/day2night/train/night/
data/processed/day2night/test/day/        # e.g. 20–50 fixed smoke-test images
data/processed/day2night/test/night/
data/processed/splits/fewshot/10shot/seed1/split.json
```

Keep the archive as one `.tar.gz` or `.zip` file in Drive. Avoid training directly from thousands of individual Drive-mounted files.

> Before sharing this notebook, push the current data-pipeline and launcher changes to `REPO_BRANCH`; the Colab VM clones from GitHub and cannot see unpushed local files.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ---- Edit these values for your team ----
REPO_URL = 'https://github.com/njzfjiang/ece-1508-project.git'
REPO_BRANCH = 'main'
DRIVE_WORKSPACE = '/content/drive/MyDrive/ece1508'
DATA_ARCHIVE = f'{DRIVE_WORKSPACE}/darkdriving_smoke.tar.gz'

# Smoke-test scope. Start with one model and a very small step count.
MODEL = 'pix2pix'          # 'pix2pix' or 'cyclegan'
SHOT = 10
SEED = 1
SMOKE_STEPS = 20
GPU_ID = 0
REBUILD_ENV = True       # Set True once if a previous Conda install was interrupted


Mounted at /content/drive


## 1. Confirm the assigned runtime

Use **Runtime → Change runtime type → GPU** before continuing. A T4 has 16 GB VRAM, so this notebook forces batch size 1 for the smoke test.

In [2]:
import os, shutil, subprocess

print(subprocess.run(['nvidia-smi'], text=True, capture_output=True).stdout)
print('VM disk usage:', shutil.disk_usage('/content'))
assert shutil.which('nvidia-smi'), 'No GPU runtime detected.'


Sun Jun 21 06:51:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the project

The repository is kept on the fast local Colab disk. Re-running this cell updates an existing checkout instead of nesting another clone.

In [3]:
from pathlib import Path

REPO_ROOT = Path('/content/ece-1508-project')
if not (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
print('Repository:', REPO_ROOT)


Repository: /content/ece-1508-project


## 3. Create the pinned Python 3.10 environment

Do not install an additional unpinned torch/diffusers/transformers stack in the Colab system Python. The project environment and upstream source revision are the reproducibility boundary.

Environment creation can take several minutes on the first run.

In [ ]:
import json, urllib.request
import subprocess # Ensure subprocess is imported
from pathlib import Path # Ensure Path is imported

MINIFORGE_ROOT = Path('/content/miniforge3')
CONDA = MINIFORGE_ROOT / 'bin' / 'conda'
if not CONDA.exists():
    installer = Path('/content/Miniforge3.sh')
    urllib.request.urlretrieve(
        'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh',
        installer,
    )
    subprocess.run(['bash', str(installer), '-b', '-p', str(MINIFORGE_ROOT)], check=True)

ENV_NAME = 'ece-1508'
ENV_PREFIX = MINIFORGE_ROOT / 'envs' / ENV_NAME
COLAB_ENV = [str(CONDA), 'run', '--no-capture-output', '-n', ENV_NAME]

def conda_env_exists():
    envs = json.loads(
        subprocess.check_output([str(CONDA), 'env', 'list', '--json'], text=True)
    )['envs']
    return str(ENV_PREFIX) in envs

def remove_env():
    if conda_env_exists():
        subprocess.run(
            [str(CONDA), 'env', 'remove', '-n', ENV_NAME, '-y'],
            check=True,
        )

def create_env():
    try:
        subprocess.run(
            [str(CONDA), 'env', 'create', '-f', str(REPO_ROOT / 'environment.yaml')],
            cwd=REPO_ROOT,
            check=True,
            capture_output=True, # Capture output
            text=True # Decode stdout/stderr as text
        )
    except subprocess.CalledProcessError as e:
        print("Conda environment creation failed!")
        print("Conda stdout:")
        print(e.stdout)
        print("Conda stderr:")
        print(e.stderr)
        raise # Re-raise the exception after printing details

def enforce_legacy_runtime_compatibility():
    # PyTorch 2.0.1 needs MKL 2024.0, while vision-aided-loss/gdown still
    # imports pkg_resources from setuptools.
    subprocess.run(
        [
            str(CONDA), 'install', '-n', ENV_NAME, '-y',
            'mkl=2024.0', 'intel-openmp=2024.0', 'setuptools<81',
        ],
        check=True,
    )

# Ensure REPO_ROOT and REBUILD_ENV are defined, assuming they are set in previous cells or globally
# For this example, assuming they are available from the kernel state.
# If not, they would need to be added here for the cell to run independently.

if REBUILD_ENV:
    remove_env()
if not conda_env_exists():
    create_env()
enforce_legacy_runtime_compatibility()

# First check whether torch imports at all. A partially created environment is
# removed and rebuilt once automatically.
import_probe = subprocess.run(
    COLAB_ENV + ['python', '-c', 'import torch; print(torch.__version__)'],
    text=True,
    capture_output=True,
)
if import_probe.returncode != 0:
    print('Existing environment failed the torch import check.')
    print(import_probe.stdout)
    print(import_probe.stderr)
    remove_env()
    create_env()
    enforce_legacy_runtime_compatibility()

runtime_probe = subprocess.run(
    COLAB_ENV + [
        'python', '-c',
        'import torch, pkg_resources; print(torch.__version__); print(pkg_resources.__file__)',
    ],
    text=True,
    capture_output=True,
)
if runtime_probe.returncode != 0:
    print(runtime_probe.stdout)
    print(runtime_probe.stderr)
    raise RuntimeError(
        'Runtime compatibility check failed: torch/pkg_resources could not import.'
    )

gpu_probe_code = '''
import torch
print('torch:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU 0:', torch.cuda.get_device_name(0))
'''
gpu_probe = subprocess.run(
    COLAB_ENV + ['python', '-c', gpu_probe_code],
    text=True,
    capture_output=True,
)
print(gpu_probe.stdout)
if gpu_probe.stderr:
    print(gpu_probe.stderr)
if gpu_probe.returncode != 0:
    subprocess.run([str(CONDA), 'list', '-n', ENV_NAME, 'torch'])
    raise RuntimeError('PyTorch failed to import inside the ece-1508 environment.')
if 'CUDA available: True' not in gpu_probe.stdout:
    subprocess.run([str(CONDA), 'list', '-n', ENV_NAME, 'torch'])
    raise RuntimeError(
        'The ece-1508 environment cannot see a CUDA GPU. Confirm that Colab '
        'is using a GPU runtime. If it is, set REBUILD_ENV=True and rerun this cell.'
    )


In [ ]:
# Pin the official img2img-turbo checkout. Dependencies were installed by environment.yaml.
setup_env = os.environ.copy()
setup_env['SKIP_INSTALL'] = '1'
setup_result = subprocess.run(
    COLAB_ENV + ['bash', 'scripts/setup_img2img_turbo.sh'],
    cwd=REPO_ROOT,
    env=setup_env,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.stderr:
    print(setup_result.stderr)
setup_result.check_returncode()


## 4. Stage the reduced dataset on the VM

The archive is copied from Drive once and extracted under the repository. Training then reads from `/content`, not the slower Drive mount.

The archive must contain the `data/processed/...` hierarchy shown at the top of this notebook.

In [ ]:
archive_on_drive = Path(DATA_ARCHIVE)
assert archive_on_drive.is_file(), f'Dataset archive not found: {archive_on_drive}'

local_archive = Path('/content') / archive_on_drive.name
if not local_archive.exists() or local_archive.stat().st_size != archive_on_drive.stat().st_size:
    shutil.copy2(archive_on_drive, local_archive)

required_split = REPO_ROOT / 'data' / 'processed' / 'splits' / 'fewshot' / f'{SHOT}shot' / f'seed{SEED}' / 'split.json'
if not required_split.is_file():
    shutil.unpack_archive(str(local_archive), str(REPO_ROOT))

required_paths = [
    required_split,
    REPO_ROOT / 'data/processed/day2night/train/day',
    REPO_ROOT / 'data/processed/day2night/train/night',
    REPO_ROOT / 'data/processed/day2night/test/day',
    REPO_ROOT / 'data/processed/day2night/test/night',
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, 'Archive has the wrong layout or is incomplete:\n' + '\n'.join(missing)
print('Dataset staged successfully.')


In [ ]:
# Inspect the exact smoke-test scope before creating the upstream view.
split = json.loads(required_split.read_text())
for relative in ['train/day', 'train/night', 'test/day', 'test/night']:
    folder = REPO_ROOT / 'data/processed/day2night' / relative
    count = sum(path.is_file() for path in folder.iterdir())
    print(f'{relative}: {count} images')
print('split metadata:', {'shot': split['shot'], 'seed': split['seed']})

subprocess.run(
    COLAB_ENV + [
        'python', 'scripts/prepare_img2img_turbo_data.py',
        '--shots', str(SHOT), '--seeds', str(SEED), '--mode', 'hardlink',
    ],
    cwd=REPO_ROOT,
    check=True,
)


## 5. Create a Colab-only smoke configuration

This leaves the checked-in `configs/base.yaml` unchanged. The smoke configuration uses batch size 1, one epoch, and a strict optimizer-step cap.

In [ ]:
SMOKE_CONFIG = REPO_ROOT / 'configs' / 'colab_smoke.yaml'
config_script = f'''from omegaconf import OmegaConf
config = OmegaConf.load(r"{REPO_ROOT / 'configs/base.yaml'}")
config.data.batch_size = 1
config.data.num_workers = 2
config.training.num_epochs = 1
config.training.max_train_steps = {SMOKE_STEPS}
config.training.save_every = {max(5, SMOKE_STEPS // 2)}
config.training.eval_every = {max(5, SMOKE_STEPS // 2)}
config.logging.use_wandb = False
OmegaConf.save(config, r"{SMOKE_CONFIG}")
'''
subprocess.run(COLAB_ENV + ['python', '-c', config_script], cwd=REPO_ROOT, check=True)
print(SMOKE_CONFIG.read_text())


## 6. Persist results to Drive

For this short smoke test, `results/` is a symlink into Drive so checkpoints survive runtime deletion. For long formal training on a rented instance, use persistent attached storage instead.

In [ ]:
drive_results = Path(DRIVE_WORKSPACE) / 'results_colab'
drive_results.mkdir(parents=True, exist_ok=True)
local_results = REPO_ROOT / 'results'
if local_results.is_symlink():
    local_results.unlink()
elif local_results.exists():
    if any(local_results.iterdir()):
        raise RuntimeError(f'{local_results} contains files; move them before replacing it with a Drive link.')
    local_results.rmdir()
local_results.symlink_to(drive_results, target_is_directory=True)
print('Results will persist at:', drive_results)


## 7. Dry-run first

This validates the selected split, upstream dataset layout, pinned checkout, and final official training command without allocating model weights.

In [ ]:
common_command = [
    'python', 'src/train/run_experiment.py',
    '--model', MODEL,
    '--shots', str(SHOT),
    '--seeds', str(SEED),
    '--gpu', str(GPU_ID),
    '--config', str(SMOKE_CONFIG),
]
subprocess.run(COLAB_ENV + common_command + ['--dry-run'], cwd=REPO_ROOT, check=True)


## 8. Run one short GPU smoke test

Expected scope: one model, one split, and `SMOKE_STEPS` optimizer steps. This is only a systems test; its generated images and losses are not experimental results.

In [ ]:
subprocess.run(COLAB_ENV + common_command, cwd=REPO_ROOT, check=True)
print('Smoke test finished. Persistent output:', drive_results / MODEL / f'{SHOT}shot' / f'seed{SEED}')


## Optional: test the other learning paradigm

After the first smoke test succeeds, change `MODEL` in the configuration cell to the other value and rerun sections 5–8. Do not launch all 18 formal experiments from this notebook.

For formal training, use a persistent GPU instance and the repository command:

```bash
DRY_RUN=1 bash scripts/run_all_experiments.sh
bash scripts/run_all_experiments.sh
```

The reduced Colab archive is suitable for smoke testing. Formal evaluation should use the agreed fixed test set and record its exact manifest.